# Decision Tree Classifier

In [1]:
import pandas as pd # type: ignore
Data_final = pd.read_csv('/Users/instructorzamora/Documents/3_Maestria_Estadistica_UNINORTE/3_Tercer_Semestre/Machine_Learning/Deteccion_Fraude/Data_final.csv')
Data_final

,D12,D14,D11,D8,TransAmt,D3,D7,dist1,dist2,V209,...,V285,id_01,D13,isFraud,card4_discover,card4_mastercard,card4_visa,card6_credit,card6_debit,card6_debit or credit
0,0.0,0.0,13.0,37.875,68.500000,13.0,0.0,19.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,1,0,0,1,0,0
1,0.0,0.0,43.0,37.875,29.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,1,0,0
2,0.0,0.0,315.0,37.875,59.000000,8.0,0.0,287.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,0,1,0,1,0
3,0.0,0.0,43.0,37.875,50.000000,0.0,0.0,8.0,37.0,0.0,...,10.0,-5.0,0.0,0.0,0,1,0,0,1,0
4,0.0,0.0,43.0,37.875,50.000000,8.0,0.0,8.0,37.0,0.0,...,0.0,0.0,0.0,0.0,0,1,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
590535,0.0,0.0,56.0,37.875,49.000000,30.0,0.0,48.0,37.0,0.0,...,1.0,-5.0,0.0,0.0,0,0,1,0,1,0
590536,0.0,0.0,0.0,37.875,39.500000,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590537,0.0,0.0,0.0,37.875,30.950001,8.0,0.0,8.0,37.0,0.0,...,0.0,-5.0,0.0,0.0,0,1,0,0,1,0
590538,0.0,0.0,22.0,37.875,117.000000,0.0,0.0,3.0,37.0,0.0,...,5.0,-5.0,0.0,0.0,0,1,0,0,1,0


El dataset final es un conjunto de datos extenso de detección de fraude con 590,540 registros y 22 columnas, diseñado para un modelo de machine learning que busca identificar transacciones fraudulentas. Contiene variables numéricas como 'TransAmt' (monto de transacción), 'dist1', 'dist2', y códigos como D12, D14, D11, junto con variables categóricas binarias que representan características de tarjetas de pago (como tipos de tarjetas Discover, Mastercard, Visa, y tipos de tarjetas de crédito/débito). La variable objetivo 'isFraud' es binaria (0 o 1), indicando si una transacción es fraudulenta, mientras que la mayoría de las otras variables son numéricas con muchos valores cercanos a cero, sugiriendo un preprocesamiento de datos previo. Este dataset parece estar preparado para entrenar un modelo de clasificación que pueda predecir la probabilidad de fraude en transacciones financieras.

In [ ]:
pip install scikit-optimize # type: ignore

## Metricas Decision Tree Classifier

In [3]:
# ------------------------
# Paso 1: Importar los paquetes necesarios
# ------------------------
import numpy as np # type: ignore
import pandas as pd 
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score
from joblib import dump
from time import time
from sklearn.model_selection import train_test_split
from skopt import BayesSearchCV
from skopt.space import Integer, Categorical

# ------------------------
# Paso 2: Cargar los datos
# ------------------------

y = Data_final['isFraud']
x = Data_final.drop(columns=['isFraud']) # anexar base de datos de JESÚS.
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.20,random_state=100,stratify=y)

# ------------------------
# Paso 3: Definir el pipeline
# ------------------------
pipe_dt = Pipeline([
    ('scaler', StandardScaler()),  # opcional para árboles, pero se mantiene por consistencia
    ('dt', DecisionTreeClassifier(random_state=42))
])

# ------------------------
# Paso 4: Espacio de búsqueda para BayesSearchCV
# ------------------------
search_spaces = {
    'dt__max_depth': Integer(3, 30),
    'dt__min_samples_split': Integer(2, 20),
    'dt__min_samples_leaf': Integer(1, 20),
    'dt__criterion': Categorical(['gini', 'entropy', 'log_loss'])
}

# ------------------------
# Paso 5: Entrenar el modelo con BayesSearchCV
# ------------------------
bayes_dt = BayesSearchCV(
    estimator=pipe_dt,
    search_spaces=search_spaces,
    n_iter=30,
    scoring='roc_auc',
    cv=5,
    n_jobs=-1,
    random_state=42
)

start_time = time()
bayes_dt.fit(x_train, y_train)
training_time_dt = time() - start_time

# Guardar el modelo
dump(bayes_dt, 'bayes_dt.joblib')

# ------------------------
# Paso 6: Hacer predicciones
# ------------------------
y_pred_dt = bayes_dt.best_estimator_.predict(x_test)
y_pred_proba_dt = bayes_dt.best_estimator_.predict_proba(x_test)[:, 1]

# ------------------------
# Paso 7: Calcular métricas
# ------------------------
precision_dt = precision_score(y_test, y_pred_dt, average='weighted')
recall_dt = recall_score(y_test, y_pred_dt, average='weighted')
accuracy_dt = accuracy_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt, average='weighted')
auc_dt = roc_auc_score(y_test, y_pred_proba_dt)

# ------------------------
# Paso 8: Resultados en DataFrame
# ------------------------
resultados_dt = pd.DataFrame({
    'Precision': [f"{precision_dt:.2f}"],
    'Recall': [f"{recall_dt:.2f}"],
    'Accuracy': [f"{accuracy_dt:.2f}"],
    'F1-Score': [f"{f1_dt:.2f}"],
    'AUC': [f"{auc_dt:.2f}"],
    'CPU time (s)': [round(training_time_dt, 2)]
})

# ------------------------
# Paso 9: Mostrar resultados
# ------------------------
print("Métricas para el modelo Decision Tree (Bayesian Optimization):")
display(resultados_dt)


Métricas para el modelo Decision Tree (Bayesian Optimization):


,Precision,Recall,Accuracy,F1-Score,AUC,CPU time (s)
0,0.96,0.97,0.97,0.96,0.81,158.81


### Análisis de las Metricas

1- Precision: El 96% de las transacciones que el modelo predijo como fraudulentas realmente lo eran.

2- Recall: Indica que el modelo identificó correctamente el 97% de todas las transacciones fraudulentas.

3- Accuracy (Exactitud): Significa que el 97% de todas las predicciones del modelo fueron correctas.

4- F1-Score: Indica que el modelo tiene un buen equilibrio entre precisión y recall, lo cual es crucial en problemas de detección de fraude donde ambos son importantes.

5- AUC (Área bajo la Curva ROC): El AUC mide la capacidad del modelo para distinguir entre las clases (fraudulentas y no fraudulentas). Un AUC de 1 indica un modelo perfecto, mientras que un AUC de 0.5 indica un modelo sin capacidad discriminativa. Un AUC de 0.81 sugiere que el modelo tiene una buena capacidad para distinguir entre transacciones fraudulentas y no fraudulentas, aunque no es perfecto.
 